# Adversarial validation (train vs test)
**tabular › validation** · notebook `adversarial_validation`

**Kapan dipakai:** Mengecek apakah data train dan test berasal dari distribusi yang sama. AUC jauh di atas 0.5 berarti ada pergeseran dan CV biasa akan menipu.

**Jebakan umum:** AUC tinggi bisa disebabkan kolom ID/tanggal/urutan baris. Telusuri fitur teratas sebelum menyimpulkan. Tidak otomatis memberi tahu cara memperbaikinya.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular/classification/titanic_train.csv"
TEST_PATH  = ROOT / "data_latihan/tabular/classification/titanic_test.csv"      # None kalau tidak ada file test
TARGET     = "survived"
ID_COL     = 'id'
DROP_COLS  = ['alive']
GROUP_COL  = None                      # isi bila ada grup agar tidak bocor
SCORING    = "f1_macro"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "adversarial_validation"
TASK       = "classification"
KIND       = "tree"
OUT = out_dir("tabular", "validation", MODEL_KEY)
set_seed(SEED)

## 2. Baca data

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
display(df.head(3))

## 3. Fitur & split

In [ ]:
from superdb.tabular import detect_columns, prepare_features, build_pipeline, split_train_valid, cv_score
cols = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS)
X, NUM, CAT = prepare_features(df, cols)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df[TARGET])
print("kelas:", dict(enumerate(le.classes_)))
groups = df[GROUP_COL] if GROUP_COL else None
Xtr, Xva, ytr, yva = split_train_valid(X, y, TASK, VALID_SIZE, SEED, groups)
GTR = None if groups is None else groups.iloc[Xtr.index]   # grup untuk CV
print(Xtr.shape, Xva.shape)

## 4. Bedakan baris train dari baris test
Label: 0 = dari train, 1 = dari test. Kalau model bisa membedakannya (AUC ≫ 0.5), distribusi berbeda dan CV biasa akan terlalu optimis.

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from superdb.tabular import feature_names
def adv_auc(A, ya, label):
    pp = build_pipeline(LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=15, random_state=SEED, n_jobs=-1, verbose=-1), NUM, CAT, kind="tree")
    pr = cross_val_predict(pp, A, ya, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
    auc = roc_auc_score(ya, pr); print(f"{label}: AUC adversarial = {auc:.3f}")
    pp.fit(A, ya)
    imp = pd.Series(pp.named_steps["model"].booster_.feature_importance("gain"), index=feature_names(pp)).sort_values(ascending=False)
    return auc, imp / imp.sum()
Xte_, _, _ = prepare_features(test, cols); Xte_ = Xte_.reindex(columns=X.columns)
A = pd.concat([X, Xte_], ignore_index=True); ya = np.r_[np.zeros(len(X)), np.ones(len(Xte_))].astype(int)
auc_real, imp_real = adv_auc(A, ya, "train vs test (data asli)")

AUC ≈ 0.5 berarti test mirip train (di sini wajar: test dipisah secara acak dari data yang sama). Sekarang kita buat **pergeseran buatan** supaya terlihat seperti apa saat ada masalah: baris kelas 3 dianggap 'test', sisanya 'train'.

In [ ]:
shift = (X["pclass"] == 3).to_numpy().astype(int)         # SINTETIK: pergeseran buatan berdasarkan kelas tiket
auc_shift, imp_shift = adv_auc(X, shift, "pergeseran buatan (kelas 3 vs lainnya)")
display(imp_shift.head(6).round(3).to_frame("share gain"))
print("fitur teratas = penyebab pergeseran:", imp_shift.index[0])

## 5. Apa yang dilakukan bila AUC tinggi?
1. Lihat fitur teratas: ID, tanggal, atau urutan baris? Buang.
2. Fitur yang memang bergeser (mis. periode berbeda): hindari bergantung padanya; buat validasi yang meniru split train/test (waktu/grup).
3. Beri bobot lebih pada baris train yang mirip test (importance weighting dari probabilitas adversarial).

In [ ]:
save_json({"auc_asli": auc_real, "auc_pergeseran_buatan": auc_shift, "fitur_teratas_shift": imp_shift.head(5).to_dict()}, OUT / "metrics.json")
print("OK. hasil di", OUT)

## Langkah berikutnya
- Pilih skema validasi SEBELUM bereksperimen, dan pakai terus untuk semua model agar skor bisa dibandingkan.
- Jika skor CV dan skor leaderboard tidak sejalan, curigai skema validasi, bukan modelnya.